# 02 - 训练数据准备

完成数据加载、ChatML格式转换、合成数据生成和数据集划分。

## 2.1 加载原始数据

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

from utils.data_utils import load_raw_data, create_sample_data
from config import DATA_DIR, DATA_CONFIG

# 加载原始数据
raw_data = load_raw_data(DATA_CONFIG['raw_data_dir'])

print(f"已加载 {len(raw_data)} 个数据子集:")
for name, samples in raw_data.items():
    print(f"  {name}: {len(samples)} 条样本")

## 2.2 查看示例数据

In [ ]:
# 查看第一条样本
first_subset = list(raw_data.keys())[0]
first_sample = raw_data[first_subset][0]

print(f"子集: {first_subset}")
print("\n问题:")
print(first_sample['instruction'])
print("\n回答:")
print(first_sample['output'][:500] + '...')

## 2.3 转换为ChatML格式

In [ ]:
from utils.data_utils import convert_to_chatml
from utils.training_utils import load_model_and_tokenizer
from config import DATA_CONFIG, BASE_MODEL, MODELS_DIR
import os

# 加载tokenizer以使用官方chat template (推荐)
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
_, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=None,  # 评测/数据处理不需要量化
    device_map='cpu',  # 数据处理用CPU即可
)

# 转换为ChatML格式 (使用tokenizer.apply_chat_template)
dataset = convert_to_chatml(
    data=raw_data,
    tokenizer=tokenizer,  # 传入tokenizer以确保格式一致性
    system_message=DATA_CONFIG['chatml']['system_message']
)

# 清理显存
import torch
torch.cuda.empty_cache()

print('数据集划分:')
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} 条")
    avg_len = sum(len(s['text']) for s in split_data) / len(split_data)
    print(f"    平均长度: {avg_len:.0f} 字符")

## 2.3b Token 长度分析

分析所有样本的 token 长度分布，提前发现可能超过模型上下文限制的样本。
超过 max_length 的样本将在训练时被静默截断，可能导致答案不完整。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

max_length = DATA_CONFIG['chatml']['max_length']

# 收集所有 split 的 token 长度
all_lengths = []
split_lengths = {}

for split_name, split_dataset in dataset.items():
    lengths = []
    for sample in split_dataset:
        text = sample['text']
        tokens = tokenizer.encode(text, add_special_tokens=False)
        lengths.append(len(tokens))
        all_lengths.append(len(tokens))
    split_lengths[split_name] = lengths

# 绘制直方图
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(all_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(max_length, color='red', linestyle='--', linewidth=2, label=f'max_length={max_length}')
plt.xlabel('Token Count')
plt.ylabel('Frequency')
plt.title('Token Length Distribution (All Splits)')
plt.legend()

plt.subplot(1, 2, 2)
for split_name, lengths in split_lengths.items():
    plt.hist(lengths, bins=30, alpha=0.5, label=split_name)
plt.axvline(max_length, color='red', linestyle='--', linewidth=2, label=f'max_length={max_length}')
plt.xlabel('Token Count')
plt.ylabel('Frequency')
plt.title('Token Length by Split')
plt.legend()

plt.tight_layout()
plt.show()

# 统计信息
print("=" * 60)
print("Token 长度统计")
print("=" * 60)
print(f"总样本数: {len(all_lengths)}")
print(f"平均长度: {np.mean(all_lengths):.0f} tokens")
print(f"中位数: {np.median(all_lengths):.0f} tokens")
print(f"最大长度: {np.max(all_lengths)} tokens")
print(f"最小长度: {np.min(all_lengths)} tokens")

over_limit = sum(1 for l in all_lengths if l > max_length)
near_limit = sum(1 for l in all_lengths if max_length * 0.8 < l <= max_length)
print(f"\n超过 {max_length} tokens (将被截断): {over_limit} 条 ({over_limit/len(all_lengths)*100:.1f}%)")
print(f"接近限制 (80%-100%): {near_limit} 条 ({near_limit/len(all_lengths)*100:.1f}%)")

if over_limit > 0:
    print(f"\n警告: {over_limit} 条样本超过上下文限制！")
    print("建议: 在 Notebook 02b 中设置更严格的质量关卡，或在 config.py 中增大 max_length")
else:
    print(f"\n所有样本均在 {max_length} tokens 限制内")
print("=" * 60)

# 按子集统计
print("\n按子集统计:")
for subset_name in set(s['subset'] for split in dataset.values() for s in split):
    subset_lengths = []
    for split in dataset.values():
        for s in split:
            if s.get('subset') == subset_name:
                tokens = tokenizer.encode(s['text'], add_special_tokens=False)
                subset_lengths.append(len(tokens))
    if subset_lengths:
        over = sum(1 for l in subset_lengths if l > max_length)
        print(f"  {subset_name}: 平均 {np.mean(subset_lengths):.0f} tokens, 最大 {np.max(subset_lengths)}, 超限 {over}")

## 2.4 查看ChatML格式示例

In [ ]:
# 查看第一条ChatML格式数据
sample = dataset['train'][0]
print("ChatML格式示例:")
print("="*60)
print(sample['text'][:1000])
print("...")
print("="*60)

## 2.5 数据验证

In [ ]:
from utils.data_utils import validate_chatml_format

# 验证数据格式
report = validate_chatml_format(dataset)

print(f"总样本数: {report['total_samples']}")
print(f"有效样本: {report['valid_samples']}")
print(f"无效样本: {report['invalid_samples']}")

if report['stats']:
    print(f"\n统计信息:")
    print(f"  平均长度: {report['stats']['avg_length']:.0f}")
    print(f"  最大长度: {report['stats']['max_length']}")
    print(f"  最小长度: {report['stats']['min_length']}")

if report['issues']:
    print(f"\n发现问题 ({len(report['issues'])} 个):")
    for issue in report['issues'][:5]:
        print(f"  - {issue}")

## 2.6 保存处理后的数据集

In [ ]:
from utils.data_utils import save_dataset

# 保存数据集
processed_dir = DATA_CONFIG['processed_data_dir']
save_dataset(dataset, processed_dir)

print(f"\n数据集已保存到: {processed_dir}")
print("文件列表:")
import os
for f in os.listdir(processed_dir):
    print(f"  {f}")

---

## 下一步

数据准备完成！接下来请打开: **03_model_benchmark.ipynb** 进行模型评测